# 📚 Build Mamba SSM from Scratch

**Interactive Colab Notebook** - Run this in Google Colab!

---

**Objectives**:
1. Learn SSM mathematical foundations
2. Implement parallel scan algorithm
3. Build selective state spaces
4. Create complete Mamba architecture

---

## 1. Setup

In [ ]:
# @title 🔧 Setup - Run this cell first!
# @markdown Install dependencies and check GPU

import subprocess
import sys

# Install PyTorch if not available
try:
    import torch
    print(f"PyTorch version: {torch.__version__}")
except ImportError:
    print("Installing PyTorch...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "-q"])
    import torch

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

## 2. SSM Mathematical Foundations

### 2.1 The Continuous-Time SSM

The fundamental equations:

$$h'(t) = Ah(t) + Bx(t)$$
$$y(t) = Ch(t)$$

Where:
- $h(t)$ = hidden state vector (dimension N)
- $x(t)$ = input
- $A$ = state transition matrix
- $B$ = input projection
- $C$ = output projection

### 2.2 Discretization (ZOH)

Convert continuous to discrete using Zero-Order Hold:

$$\bar{A} = e^{A\Delta}$$
$$\bar{B} = (e^{A\Delta} - I)A^{-1}B\Delta$$

Discrete recurrence:
$$h_k = \bar{A}h_{k-1} + \bar{B}x_k$$

In [ ]:
# @title 🧮 Basic Sequential SSM Implementation

class BasicSSM(nn.Module):
    """Basic State Space Model - Sequential computation"""
    
    def __init__(self, d_model, d_state):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        
        # A: diagonal matrix, parameterized by log-eigenvalues
        self.A_log = nn.Parameter(torch.randn(d_state))
        
        # Projections
        self.B_proj = nn.Linear(d_model, d_state)
        self.C_proj = nn.Linear(d_state, d_model)
        
        # Delta (step size)
        self.delta = nn.Parameter(torch.ones(d_state) * 0.1)
    
    def forward(self, x):
        batch, seq_len, _ = x.shape
        
        # Discretize A
        A_bar = torch.exp(self.A_log * self.delta)
        
        # Discretization factor: (exp(A*Δ) - 1) / A
        disc = torch.where(
            torch.abs(self.A_log) > 1e-6,
            (A_bar - 1) / self.A_log,
            self.delta
        )
        
        # Sequential computation
        h = torch.zeros(batch, self.d_state, device=x.device)
        outputs = []
        
        for t in range(seq_len):
            B_t = self.B_proj(x[:, t])
            B_bar = B_t * disc * self.delta
            h = A_bar.unsqueeze(0) * h + B_bar
            outputs.append(self.C_proj(h))
        
        return torch.stack(outputs, dim=1)

# Test
print("Testing Basic SSM...")
ssm = BasicSSM(d_model=64, d_state=128).to(device)
x = torch.randn(2, 16, 64).to(device)
y = ssm(x)
print(f"Input: {x.shape} → Output: {y.shape}")
print(f"Parameters: {sum(p.numel() for p in ssm.parameters()):,}")

## 3. Parallel Scan Algorithm

### The Key Insight

The SSM recurrence $h_k = A_h_{k-1} + Bx_k$ is **associative**:

$$(a \oplus b) \oplus c = a \oplus (b \oplus c)$$

This allows **parallel computation** in O(log n) instead of O(n)!

In [ ]:
# @title ⚡ Parallel Scan Implementation

def parallel_scan(els):
    """
    Parallel scan for associative operation.
    Operation: (a, b) ⊕ (c, d) = (a*c, a*d + b)
    """
    batch, seq_len, d_state = els.shape
    
    # Pad to power of 2
    n = 1 << (seq_len - 1).bit_length()
    padded = torch.zeros(batch, n, d_state, device=els.device, dtype=els.dtype)
    padded[:, :seq_len, :] = els
    
    # Parallel scan (bottom-up)
    for stride in range(n.bit_length() - 1):
        step = 1 << stride
        
        # Even positions
        even = padded[:, ::step*2, :]
        # Odd positions (shifted)
        odd = padded[:, step::step*2, :]
        
        # Combine: (a, b) ⊕ (c, d) = (a*c, a*d + b)
        combined = torch.cat([
            even[:, :1, :],
            even[:, 1:, :] * odd + even[:, :-1, :]
        ], dim=1)
        
        # Update odd positions
        padded[:, step::step*2, :] = combined[:, 1:, :] if combined.shape[1] > 1 else combined
    
    return padded[:, :seq_len, :]


def ssm_parallel_scan(A_bar, B_bar):
    """Parallel scan for SSM with diagonal A"""
    batch, seq_len, d_state = B_bar.shape
    
    # Pad to power of 2
    n = 1 << (seq_len - 1).bit_length()
    
    A = torch.zeros(batch, n, d_state, device=A_bar.device, dtype=A_bar.dtype)
    B = torch.zeros(batch, n, d_state, device=B_bar.device, dtype=B_bar.dtype)
    A[:, :seq_len, :] = A_bar
    B[:, :seq_len, :] = B_bar
    
    # Parallel scan
    for stride in range(n.bit_length() - 1):
        step = 1 << stride
        
        A_even = A[:, ::step*2, :]
        B_even = B[:, ::step*2, :]
        A_odd = A[:, step::step*2, :]
        B_odd = B[:, step::step*2, :]
        
        # Combine: (a_l, b_l) ⊕ (a_r, b_r) = (a_l*a_r, a_l*b_r + b_l)
        A[:, step::step*2, :] = A_even * A_odd
        B[:, step::step*2, :] = A_even * B_odd + B_even
    
    return B[:, :seq_len, :]

print("✅ Parallel scan defined!")

In [ ]:
# @title 🧪 Test Parallel Scan

# Test with simple addition
x = torch.tensor([[[1.], [2.], [3.], [4.], [5.], [6.], [7.], [8.]]])
result = parallel_scan(x)
print("Input:  ", x[0, :, 0].tolist())
print("Result: ", result[0, :, 0].tolist())
print("Expected:", [1, 3, 6, 10, 15, 21, 28, 36])

# Verify
assert torch.allclose(result[0, :, 0], torch.tensor([1, 3, 6, 10, 15, 21, 28, 36]))
print("✅ Parallel scan works!")

## 4. Selective SSM (Mamba's Innovation)

### The Key Innovation

Make parameters **input-dependent**:

$$B_k = \text{Linear}_B(x_k)$$
$$C_k = \text{Linear}_C(x_k)$$
$$\Delta_k = \text{softplus}(\text{Linear}_\Delta(x_k))$$

This allows selective attention to content!

In [ ]:
# @title 🎯 Selective SSM Implementation

class SelectiveSSM(nn.Module):
    """Mamba's Selective State Space Model"""
    
    def __init__(self, d_model, d_state, dt_rank=16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.dt_rank = dt_rank
        
        # A: state dynamics (not selective)
        self.A_log = nn.Parameter(torch.randn(d_state))
        
        # D: skip connection
        self.D = nn.Parameter(torch.ones(d_state))
        
        # Input to selective params (Δ, B, C)
        self.x_proj = nn.Linear(d_model, dt_rank + d_state * 2, bias=False)
        
        # Output projection
        self.out_proj = nn.Linear(d_state, d_model, bias=False)
    
    def forward(self, x):
        batch, seq_len, d_model = x.shape
        d_state = self.d_state
        
        # Compute selective parameters
        x_params = self.x_proj(x)
        
        # Split: Δ, B, C
        delta = x_params[:, :, :self.dt_rank]
        B = x_params[:, :, self.dt_rank:self.dt_rank + d_state]
        C = x_params[:, :, self.dt_rank + d_state:]
        
        # Δ with softplus (positive step size)
        delta = F.softplus(delta)
        
        # Discretize A with selective Δ
        A_bar = torch.exp(self.A_log.unsqueeze(0).unsqueeze(0) * delta.unsqueeze(-1))
        
        # Discretize B
        disc = (A_bar - 1) / torch.where(
            torch.abs(self.A_log.unsqueeze(0).unsqueeze(0)) > 1e-6,
            self.A_log.unsqueeze(0).unsqueeze(0),
            1e-6
        )
        B_bar = disc * B.unsqueeze(-1)
        
        # Parallel scan
        h = ssm_parallel_scan(A_bar, B_bar)
        
        # Output: y = C * h + D * x
        y = torch.einsum('bldn,bln->bld', h, C)
        y = y + x * self.D.unsqueeze(0).unsqueeze(0)
        
        return self.out_proj(y)

# Test
print("Testing Selective SSM...")
ssm = SelectiveSSM(d_model=64, d_state=128).to(device)
x = torch.randn(2, 32, 64).to(device)
y = ssm(x)
print(f"Input: {x.shape} → Output: {y.shape}")

# Check gradients
loss = y.sum()
loss.backward()
print("✅ Backpropagation works!")

## 5. Full Mamba Block

In [ ]:
# @title 🏗️ Complete Mamba Block

class MambaBlock(nn.Module):
    """Full Mamba Block with gating and normalization"""
    
    def __init__(self, d_model, d_state=128, expand=2, dt_rank=16):
        super().__init__()
        self.d_model = d_model
        self.d_inner = int(expand * d_model)
        self.d_state = d_state
        self.dt_rank = dt_rank
        
        # Pre-norm
        self.norm = nn.LayerNorm(d_model)
        
        # Input projection (expand)
        self.in_proj = nn.Linear(d_model, self.d_inner * 2)
        
        # Selective SSM
        self.ssm = SelectiveSSM(self.d_inner, d_state, dt_rank)
        
        # Output projection (contract)
        self.out_proj = nn.Linear(self.d_inner, d_model)
    
    def forward(self, x):
        # Pre-norm
        x_norm = self.norm(x)
        
        # Input projection with split for gating
        x_inner, gate = self.in_proj(x_norm).chunk(2, dim=-1)
        
        # SiLU gating
        gate = F.silu(gate)
        
        # SSM
        ssm_out = self.ssm(gate)
        
        # Apply gate
        y = ssm_out * gate
        
        # Output projection
        y = self.out_proj(y)
        
        # Residual
        return y + x

# Test
print("Testing Mamba Block...")
block = MambaBlock(d_model=256, d_state=128, expand=2).to(device)
x = torch.randn(2, 32, 256).to(device)
y = block(x)
print(f"Input: {x.shape} → Output: {y.shape}")
print(f"Parameters: {sum(p.numel() for p in block.parameters()):,}")

## 6. Benchmark

In [ ]:
# @title ⏱️ Benchmark: Mamba vs Transformer

class TransformerBlock(nn.Module):
    """Standard Transformer block for comparison"""
    def __init__(self, d_model, n_heads=8):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model)
        )
    
    def forward(self, x):
        x = x + self.attn(self.norm(x), self.norm(x), self.norm(x))[0]
        x = x + self.ff(self.norm(x))
        return x


# Benchmark function
def benchmark(model, seq_len, n_runs=10):
    x = torch.randn(4, seq_len, 256).to(device)
    
    # Warm up
    with torch.no_grad():
        _ = model(x)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs):
            y = model(x)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    return (time.perf_counter() - start) / n_runs * 1000


seq_lens = [64, 128, 256, 512, 1024]
mamba_times = []
transformer_times = []

print("Running benchmarks...\n")
print("Seq Len | Mamba (ms) | Transformer (ms) | Speedup")
print("-" * 55)

for seq_len in seq_lens:
    mamba = MambaBlock(d_model=256, d_state=128).to(device)
    transformer = TransformerBlock(d_model=256).to(device)
    
    t_mamba = benchmark(mamba, seq_len)
    t_trans = benchmark(transformer, seq_len)
    
    mamba_times.append(t_mamba)
    transformer_times.append(t_trans)
    
    print(f"{seq_len:7} | {t_mamba:9.2f} | {t_trans:16.2f} | {t_trans/t_mamba:.2f}x")

print("\n✅ Benchmark complete!")

In [ ]:
# @title 📊 Plot Results

plt.figure(figsize=(10, 5))
plt.plot(seq_lens, mamba_times, 'o-', label='Mamba', linewidth=2, markersize=8)
plt.plot(seq_lens, transformer_times, 's-', label='Transformer', linewidth=2, markersize=8)
plt.xlabel('Sequence Length')
plt.ylabel('Time (ms)')
plt.title('Forward Pass Speed: Mamba vs Transformer')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print("\n📈 Mamba shows better scaling with longer sequences!")

## 7. Summary

### What we built:

| Component | Description |
|-----------|-------------|
| **Basic SSM** | Mathematical foundation |
| **Parallel Scan** | O(log n) computation |
| **Selective SSM** | Input-dependent parameters |
| **Mamba Block** | Full architecture with gating |

### Key Insights:

1. **SSMs** compress sequences into state vectors
2. **Parallel scan** enables efficient computation
3. **Selection** allows content-based reasoning
4. **Mamba** matches transformers with O(N) complexity

---

**Next Steps:**
- Add more layers for language model
- Train on text data
- Compare with hybrids (Mamba + Attention)

🎉 **Congratulations! You built Mamba from scratch!**